# 01 — Bronze Ingestion (thin wrapper)

Incremental, append-only CSV → Delta ingestion into `workspace.bronze.*`.
All heavy lifting lives in `src/transforms.py`; this notebook only wires
paths, the run-log watermark and the explicit-column INSERT.

Bronze keeps ALL raw rows (duplicates included) — deduplication happens in
Silver. No MERGE and no schema changes ever in the raw layer.

In [0]:
# --- Bootstrap: make the repo's src/ package importable + config ---
import sys

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])  # .../notebooks/01_bronze -> repo root
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

import datetime

from src.transforms import BUSINESS_COLUMNS, new_rows_only, read_csv_all_strings

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

LANDING_PATH = "/Volumes/workspace/landing/raw_files"
BRONZE_SCHEMA = "workspace.bronze"
RUN_LOG_TABLE = "workspace.gold.pipeline_run_log"
DATASETS = [("studies", "studies.csv"), ("sites", "sites.csv"),
            ("subjects", "subjects.csv"), ("visits", "visits.csv")]

print(f"Bronze ingestion starting | repo root: {_root} | source: {LANDING_PATH}")

In [0]:
# --- Watermark + incremental append-only ingest (no MERGE, no schema changes) ---
# Watermark = end_time of the last successful Bronze_Ingestion run in the
# run log (the bronze tables themselves carry no timestamps). Only files
# newer than the watermark are processed; only rows not already present
# are appended (null-safe anti-join from src/transforms.py).
watermark = spark.sql(f"""
    SELECT MAX(end_time) AS wm FROM {RUN_LOG_TABLE}
    WHERE stage_name = 'Bronze_Ingestion' AND status = 'Success'
""").collect()[0]["wm"]
print(f"Watermark: {watermark}")

for table, fname in DATASETS:
    path = f"{LANDING_PATH}/{fname}"
    cols = BUSINESS_COLUMNS[table]
    spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_SCHEMA}.{table} "
              f"({', '.join(c + ' STRING' for c in cols)}) USING DELTA")
    mtime = datetime.datetime.fromtimestamp(
        dbutils.fs.ls(path)[0].modificationTime / 1000.0, tz=datetime.UTC)
    if watermark is not None and mtime <= watermark:
        print(f"{table}: SKIP (file unchanged since last successful run)")
        continue
    src_df = read_csv_all_strings(spark, path, cols)
    csv_rows = src_df.count()
    new_df = new_rows_only(src_df, spark.table(f"{BRONZE_SCHEMA}.{table}"), cols)
    n_new = new_df.count()
    if n_new > 0:
        # explicit column-list INSERT: never touches _rescued_data / load_timestamp
        col_sql = ", ".join(cols)
        new_df.createOrReplaceTempView("v_bronze_new")
        spark.sql(f"INSERT INTO {BRONZE_SCHEMA}.{table} ({col_sql}) "
                  f"SELECT {col_sql} FROM v_bronze_new")
    bronze_rows = spark.table(f"{BRONZE_SCHEMA}.{table}").count()
    print(f"{table}: csv={csv_rows} new={n_new} bronze={bronze_rows}")

In [0]:
# --- Reconciliation: bronze Delta counts (idempotency evidence) ---
print("=" * 60)
print("=== BRONZE TABLE RECORD COUNTS ===")
print("=" * 60)
total = 0
for table, _ in DATASETS:
    n = spark.table(f"{BRONZE_SCHEMA}.{table}").count()
    total += n
    print(f"{BRONZE_SCHEMA}.{table}: {n} rows")
print(f"TOTAL: {total} rows — Bronze ingestion complete.")